# Model Retraining Strategies

---

In this notebook, we will learn **when** and **how** to retrain a deployed model to recover from drift or improve performance over time.

We will cover:

- Why retraining is necessary
- The three main retraining strategies (scheduled, triggered, online)
- The retraining workflow from detection to deployment
- Shadow deployment and A/B testing for safe rollouts
- Practical retraining patterns for freelancers

---

### 1. Why Retrain?

From the previous notebook, we know that models degrade because the world changes. Retraining updatest the model's learned patterns to reflect the current state of the data.

But retraining isn't free:

| **Cost** | **Description** |
| :--- | :--- |
| **Data collection** | You need recent, labeled data - which may require human annotation. |
| **Compute** | Training costs time and resources (especially for large models). |
| **Validation** | You must verify the new model is actually better than the old one before deploying it. |
| **Risk** | A bad retrain can make things worse if the new data has issues. |

So the question isn't *"Should I retrain?"* - it's *"When is it worth the cost?"*

---

## 2. Retraining Strategies

### 2.1. Scheduled Retraining

**When:** At fixed intervals (daily, weekly, monthly). **For example,** every Monday at 2am:
1. Pull the latest labeled data
2. Train a new model
3. Validate it against a holdout set
4. If performance meets threshold → deploy
5. If not → keep the old model, alert the team

| **Pros** | **Cons** |
| :--- | :--- |
| Simple to implement (cron job / GitHub Actions schedule) | May retrain when nothing has changed (wasted compute) |
| Predictable and easy to monitor | May not retrain fast enough if drift happens between intervals |

**Best for:** Models where data accumulates steadily and drift is gradual. Most freelance projects.

### 2.2. Triggered Retraining

**When:** Only when a drift or performance signal exceeps a threshold. **For example**, monitor daily:
- If PSI > 0.2 for any feature → trigger retraining
- If accuracy < 0.85 on validation set → trigger retraining
- If avg. confidence < 0.70 for 3 consecutive days → trigger retraining

| **Pros** | **Cons** |
| :--- | :--- |
| Only retraing when needed (efficient) | Requires monitoring ingrastructure |
| Responds to drift quickly | Trigger thresholds need tuning |

**Best for:** Production systems with monitoring in place. Combines well with scheduled retraining as a fallback.

### 2.3. Online Learning (Incremental)

**When:** Continuously, as new data arrives. **For example**, for each new labeled sample:
1. Update the model incrementally (partial fit)
2. The model continuously adapts

| **Pros** | **Cons** |
| :--- | :--- |
| Fasted adaptation to change | Not all algorithms support it |
| No retraining pipeline needed | Risk of catastrophic forgetting |
|  | Hard to validate and roll back |

**Best for:** Specialized use cases (recommendation systems, ad bidding). Rarely used for standard classification/regression in freelance settings.

### 2.4. Comparison

| Strategy | Complexity | Data Needs | Best For |
| :--- | :---: | :--- | :--- |
| **Scheduled** | Low | Batch of labeled data at each interval | Most projects |
| **Triggered** | Medium | Monitoring data + labeled data | Production with monitoring |
| **Online** | High | Continuous labeled stream | Specialized systems |

---

## 3. The Retraining Workflow

Regardless of the trigger, the retraining process follows a consistent workdlow:

1. **COLLECT:** Gather new labeled data
2. **TRAIN:** Train a new model (same pipeline, new data)
3. **EVALUATE:** Compare new model vs. current model on a holdout set
4. **DECIDE:** Is the new model better? (accuracy, F1, business metric)
5. **DEPLOY:** Replace the old model (or shadow deploy first)
6. **MONITOR:** Watch the new model's performance

> ⚠️ **Step 3 is critical!** Never deploy a retrained model without comparing it to the current one.

In [ ]:
from sklearn.metrics import accuracy_score

# Evaluate both models on the same holdout set
current_accuracy = accuracy_score(y_holdout, current_model.predict(X_holdout))
new_accuracy = accuracy_score(y_holdout, new_model.predict(X_holdout))

print(f"Current model: {current_accuracy:.4f}")
print(f"New model:     {new_accuracy:.4f}")

if new_accuracy > current_accuracy:
    print("✅ New model is better → deploy")
    joblib.dump(new_model, "models/iris_pipeline.joblib")
else:
    print("❌ New model is worse → keep current model")

This simple comparison prevents deploying a model that's actually worse, which can happen if the new training data has quality issues.

---

## 4. Safe Deployment Patterns

### 4.1. Shadow Deployment

Run the new model **alongside** the current one. Both process every request, but only the current model's predictions are served to users. You compare their outputs:

```
Request → Current Model → serves prediction to user
       → New Model     → logs prediction (not served)
```

After a week, compare the logs:
- Do they agree most of the time? → Safe to switch.
- Do they disagree significantly? → Investigate before switching.

**Pros:** Zero risk to users. **Cons:** Requires running two models.

### 4.2. A/B Testing

Route a small percentage of traffic (e.g., 10%) to the new model and 90% to the current one. Compare real-world performance:

```
Request → Router → 90% → Current Model
                 → 10% → New Model
```

If the new model performs better on its 10% of traffic, gradually increase to 100%.

**Pros:** Tests with real users. **Cons:** Some users get potentially worse model.

### 4.3. For Freelancers: Keep It Simple

Shadow deployment and A/B testing are enterprise patterns. For most freelance projects, the practical approach is:

1. **Retrain locally** with new data
2. **Evaluate on a holdout set** (automated or manual)
3. **If better → replace the .joblib file** and redeploy
4. **If worse → don't deploy** and investigate the data

Simple, effective, and takes 30 minutes.

---

## 5. Automating Retraining with GitHub Actions

For scheduled retraining, you can use a GitHub Actions workflow on a cron schedule:

```yaml
# .github/workflows/retrain.yml
name: Scheduled Model Retraining

on:
  schedule:
    - cron: '0 2 * * 1'   # Every Monday at 2am UTC
  workflow_dispatch:       # Also allow manual trigger

jobs:
  retrain:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.13'

      - name: Install dependencies
        run: pip install scikit-learn joblib numpy

      - name: Retrain and evaluate
        run: python scripts/retrain.py

      - name: Commit updated model (if improved)
        run: |
          git config user.name "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"
          git add models/iris_pipeline.joblib
          git diff --staged --quiet || git commit -m "chore: retrained model [automated]"
          git push
```

The `retrain.py` script would:
1. Load the latest data
2. Train a new pipeline
3. Evaluate against the current model on a holdout set
4. Only save the new model if it's better

Combined with Railway's auto-deploy, pushing the updated model file triggers a new deployment automatically.

---

## 6. Model Versioning

When you retrain, keep trach of which model is deployed:

In [ ]:
metadata = {
    "model_name": "iris_pipeline",
    "model_version": "2.0",            # Incremented from 1.0
    "training_date": "2026-03-11",
    "training_data_size": 250,           # Larger than v1.0 (150)
    "test_accuracy": 0.98,
    "replaced_version": "1.0",
    "reason": "Scheduled retrain with 100 new labeled samples."
}

Save this alongside the model (just like we did in the Model Persistence section). This metadatta is your **audit trail**. It tells you:
- Which version is currently deployed
- When it was trained
- Why it was retrained
- How it compared to the previous versions

---

## 7. Summary

| **Concept** | **Key Takeaway** |
| :--- | :--- |
| **Scheduled retraining** | Retrain at fixed intervals (weekly, monthly). Simple and predictable. Best default for freelancers. |
| **Triggered retraining** | Retrain when drift of performance signals cross a threshold. More efficient, requires monitoring. |
| **Online learning** | Continuous updates. Complex, specialized use cases only. |
| **Always compare** | Never deploy a retrained model without evaluating it against the current one on a holdout set. |
| **Shadow deployment** | Run new model alongside old one to compare without risk. Enterprise pattern. |
| **GitHub Actions cron** | Automate scheduled retraining with a cron-triggered workflow. Push the new model → Railway auto-deploys. |
| **Model versioning** | Save metadata with every model: version, date, accuracy, reason for retraining. |

